In [1]:
!pip install -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.0/755.0 kB 13.1 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.91.0
    Uninstalling openai-1.91.0:
      Successfully uninstalled openai-1.91.0


In [2]:
from google.colab import userdata
import os

# Colab에 저장한 Secret에서 API 키 가져와 환경 변수에 등록
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [3]:
import openai
client = openai.OpenAI()

##실습 5.1. Chain of Thought 기법 실습

In [4]:
# Chain-of-Thought 없이 답변 생성
question = "철수는 사과 10개를 가지고 있었는데, 그 중 3개를 먹은 후 5개를 더 구입했습니다. 철수가 현재 가지고 있는 사과는 몇 개일까요?"
prompt1 = question  # 별도 지시 없이 바로 질문만 던집니다.
response1 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": prompt1}
    ],
    model="gpt-4o-mini",
    temperature=0
)
print("CoT 미적용 답변:", response1.choices[0].message.content)

CoT 미적용 답변: 철수는 처음에 사과 10개를 가지고 있었습니다. 그 중 3개를 먹었으므로 남은 사과는 10 - 3 = 7개입니다. 이후 5개를 더 구입했으므로 현재 가지고 있는 사과는 7 + 5 = 12개입니다. 

따라서 철수가 현재 가지고 있는 사과는 12개입니다.


In [5]:
# Chain-of-Thought 프롬프트와 함께 답변 생성
cot_prompt = "(생각을 단계별로 진행합니다)\n" + question  # 단계적 사고를 지시하는 문구를 추가
response2 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": cot_prompt}
    ],
    model="gpt-4o-mini",
    temperature=0
)
print("CoT 적용 답변:", response2.choices[0].message.content)

CoT 적용 답변: 단계별로 생각해보겠습니다.

1. 철수가 처음에 가지고 있던 사과의 개수: 10개
2. 철수가 먹은 사과의 개수: 3개
3. 철수가 사과를 먹은 후 남은 사과의 개수: 10개 - 3개 = 7개
4. 철수가 추가로 구입한 사과의 개수: 5개
5. 철수가 현재 가지고 있는 사과의 총 개수: 7개 + 5개 = 12개

따라서, 철수가 현재 가지고 있는 사과는 12개입니다.


##5.2. Self-Consistency 기법 실습

In [6]:
import collections

question = "어떤 수를 3배한 결과에 4를 더하면 19가 됩니다. 그 수는 무엇일까요? 단계별로 풀어보세요."
messages = [{"role": "user", "content": question}]
answers = []

# 동일한 질문에 대해 여러 번 응답 생성 (예: 5회)
for i in range(5):
    response = client.chat.completions.create(
        messages=messages,
        model="gpt-4o-mini",
        temperature=1.0
    )
    answer = response.choices[0].message.content.strip()
    answers.append(answer)
    print(f"응답{i+1}: {answer}")

# 최종 답 도출 – 모든 응답의 마지막 줄(결과)을 모아 다수결 투표
final_answers = [ans.split()[-1] for ans in answers]  # 각 답변에서 마지막 단어를 가져와서 (여기서는 '5'같은 숫자일 것이라 가정)
counter = collections.Counter(final_answers)
final_answer = counter.most_common(1)[0][0]  # 가장 빈도 높은 답
print("\n다수결에 따른 최종 답:", final_answer)

응답1: 주어진 문제를 식으로 표현해 보겠습니다. 어떤 수를 \( x \)라고 가정하겠습니다. 그러면 "3배한 결과에 4를 더하면 19가 됩니다"라는 문장은 다음과 같은 방정식으로 표현할 수 있습니다:

\[
3x + 4 = 19
\]

이제 이 방정식을 단계별로 풀어보겠습니다.

1. **양변에서 4를 뺍니다.**

\[
3x + 4 - 4 = 19 - 4
\]

이렇게 하면:

\[
3x = 15
\]

2. **양변을 3으로 나눕니다.**

\[
\frac{3x}{3} = \frac{15}{3}
\]

즉, 

\[
x = 5
\]

따라서, 원하는 수는 \( \boxed{5} \)입니다.
응답2: 주어진 문제를 단계별로 풀어보겠습니다.

1. **문제 이해하기**:
   문제는 "어떤 수를 3배한 결과에 4를 더하면 19가 된다"는 것입니다. 이를 수식으로 표현하면 다음과 같습니다.
   \[
   3x + 4 = 19
   \]
   여기서 \(x\)는 찾고자 하는 수입니다.

2. **수식 변형하기**:
   먼저, 4를 양쪽에서 빼서 수식을 변형합니다.
   \[
   3x + 4 - 4 = 19 - 4
   \]
   따라서
   \[
   3x = 15
   \]

3. **x의 값을 구하기**:
   이제 양쪽을 3으로 나눠서 \(x\) 값을 구합니다.
   \[
   x = \frac{15}{3} = 5
   \]

4. **답 확인하기**:
   \(x = 5\)인지를 확인하기 위해 원래의 수식에 대입해보겠습니다.
   \[
   3(5) + 4 = 15 + 4 = 19
   \]
   확인 결과, 맞습니다.

결론적으로, 찾고자 하는 수는 **5**입니다.
응답3: 주어진 문제를 수식으로 나타내겠습니다. \( x \)를 구하고자 하는 수라고 하겠습니다. 문제를 수식으로 표현하면 다음과 같습니다:

\[
3x + 4 = 19
\]

이제 단계별로 풀어보겠습니다.

**1단계: 4를 양변에서 빼기**

먼저, 4를 양쪽에서 빼줍니다.

\[

##API를 이용한 Iteration : 1번의 요청에 5개의 응답

In [ ]:
import collections

question = "어떤 수를 3배한 결과에 4를 더하면 19가 됩니다. 그 수는 무엇일까요? 단계별로 풀어보세요."
messages = [{"role": "user", "content": question}]
answers = []

# 동일한 질문에 대해 여러 번 응답 생성 (예: 5회)
response = client.chat.completions.create(
    messages=messages,
    model="gpt-4o-mini",
    temperature=1.0,
    n=5
)
for i in range(5):
    answer = response.choices[i].message.content.strip()
    answers.append(answer)
    print(f"응답{i+1}: {answer}")

# 최종 답 도출 – 모든 응답의 마지막 줄(결과)을 모아 다수결 투표
final_answers = [ans.split()[-1] for ans in answers]  # 각 답변에서 마지막 단어를 가져와서 (여기서는 '5'같은 숫자일 것이라 가정)
counter = collections.Counter(final_answers)
final_answer = counter.most_common(1)[0][0]  # 가장 빈도 높은 답
print("\n다수결에 따른 최종 답:", final_answer)

응답1: 문제를 단계별로 풀어보겠습니다.

1. **문제를 이해하기**:
   어떤 수를 \( x \)라고 하겠습니다. 문제에서 "어떤 수를 3배한 결과에 4를 더하면 19가 됩니다"라고 했으므로, 이 내용을 수식으로 표현할 수 있습니다.

2. **수식 만들기**:
   \( 3x + 4 = 19 \)

3. **방정식 풀기**:
   이제 이 방정식을 풀어보겠습니다.

   - 첫 번째 단계로, 양변에서 4를 빼줍니다:
     \[
     3x + 4 - 4 = 19 - 4
     \]
     \[
     3x = 15
     \]

   - 두 번째 단계로, 양변을 3으로 나누어 \( x \)를 구합니다:
     \[
     x = \frac{15}{3}
     \]
     \[
     x = 5
     \]

4. **결과 확인**:
   이제 \( x = 5 \)가 맞는지 확인해 보겠습니다. 

   - 5를 3배하면: \( 3 \times 5 = 15 \)
   - 여기에 4를 더하면: \( 15 + 4 = 19 \)

   확인 결과, 주어진 조건을 만족합니다.

따라서, 그 수는 \( 5 \)입니다.
응답2: 문제를 단계별로 풀어보겠습니다.

1. 문제를 식으로 표현해 보겠습니다. 어떤 수를 \( x \)라고 할 때, 문제의 조건에 따라 다음과 같은 식을 세울 수 있습니다:
   \[
   3x + 4 = 19
   \]

2. 이제 이 식에서 \( x \)를 구하기 위해 먼저 양변에서 4를 빼줍니다:
   \[
   3x + 4 - 4 = 19 - 4
   \]
   \[
   3x = 15
   \]

3. 다음으로, 양변을 3으로 나누어서 \( x \)를 구합니다:
   \[
   \frac{3x}{3} = \frac{15}{3}
   \]
   \[
   x = 5
   \]

따라서, 그 수는 \( 5 \)입니다.
응답3: 문제를 단계별로 풀어보겠습니다.

1. **문제 이해하기**: 어떤 수를 \( x \)라고 하

##실습5.3. Reflexion 기법 산술문제 실습

In [7]:
# 1. 초기 질문에 대한 1차 답변 생성
question = "7의 2승에 5를 곱한 값은 무엇인가?"
response1 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": question}
    ],
    model="gpt-4o-mini",
    temperature=0
)
answer1 = response1.choices[0].message.content.strip()
print("1차 답변:", answer1)

# 2. 1차 답변에 대한 피드백 생성 (모델 스스로에게 해볼 수도 있지만, 여기서는 정답 알고 있다고 가정하고 직접 피드백 작성)
correct_answer = 245
if str(correct_answer) in answer1:
    feedback = "정답입니다. 잘 해결했어요!"
    need_retry = False
else:
    feedback = "오답입니다. 계산을 다시 해보세요. (힌트: 7의 제곱값을 정확히 구한 후 곱하세요.)"
    need_retry = True

print("피드백:", feedback)

1차 답변: 7의 2승은 \( 7^2 = 49 \)입니다. 여기에 5를 곱하면:

\[ 49 \times 5 = 245 \]

따라서, 7의 2승에 5를 곱한 값은 245입니다.
피드백: 정답입니다. 잘 해결했어요!


In [9]:
# 3. 피드백을 포함한 새로운 프롬프트 구성하여 2차 답변 생성 (만약 재시도가 필요한 경우)
if need_retry:
    retry_prompt = f"이전 답변: {answer1}\n피드백: {feedback}\n따라서 답을 다시 구해보세요."
    response2 = client.chat.completions.create(
        messages=[
            {"role": "user", "content": retry_prompt}
        ],
        model="gpt-4o-mini",
        temperature=0
    )
    answer2 = response2.choices[0].message.content.strip()
    print("2차 답변:", answer2)

##실습5.4. Reflexion 기법 에세이작성 실습

In [10]:
# 에세이 작성 1차 시도
prompt = "기후 변화의 원인과 해결 방안에 대해 간략한 에세이를 작성하시오."
response_essay1 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": prompt}
    ],
    model="gpt-4o-mini",
    temperature=0
)
essay1 = response_essay1.choices[0].message.content
print("1차 에세이:\n", essay1)

1차 에세이:
 기후 변화는 현대 사회가 직면한 가장 심각한 환경 문제 중 하나로, 지구의 평균 기온 상승, 해수면 상승, 극단적인 기상 현상 등의 형태로 나타나고 있습니다. 이러한 기후 변화의 주된 원인은 인간 활동에 의해 발생하는 온실가스의 증가입니다. 특히, 화석 연료의 연소, 산업 활동, 농업 및 임업 등에서 발생하는 이산화탄소(CO2), 메탄(CH4), 아산화질소(N2O) 등의 온실가스가 대기 중에 축적되어 지구의 열을 가두는 효과를 만들어냅니다.

기후 변화의 해결 방안은 여러 가지가 있지만, 가장 중요한 것은 온실가스 배출을 줄이는 것입니다. 이를 위해 첫째, 재생 가능 에너지의 사용을 확대해야 합니다. 태양광, 풍력, 수력 등 청정 에너지원으로의 전환은 화석 연료 의존도를 줄이고, 온실가스 배출을 감소시키는 데 기여할 수 있습니다. 둘째, 에너지 효율성을 높이는 기술을 개발하고 보급해야 합니다. 건물, 교통수단, 산업 공정에서 에너지를 절약하는 방법을 도입함으로써 불필요한 에너지 소비를 줄일 수 있습니다.

셋째, 지속 가능한 농업과 임업 practices를 통해 탄소 흡수 능력을 높이는 것도 중요합니다. 나무를 심고, 토양의 건강을 유지하며, 화학 비료 사용을 줄이는 등의 방법은 자연 생태계를 회복하고 기후 변화에 대한 저항력을 높이는 데 도움이 됩니다. 마지막으로, 개인과 지역 사회의 인식 개선과 행동 변화도 필수적입니다. 기후 변화에 대한 교육과 캠페인을 통해 사람들의 인식을 높이고, 일상생활에서의 작은 실천들이 모여 큰 변화를 이끌어낼 수 있습니다.

결론적으로, 기후 변화는 인류의 지속 가능한 미래를 위협하는 중대한 문제입니다. 이를 해결하기 위해서는 정부, 기업, 개인이 함께 협력하여 온실가스 배출을 줄이고, 지속 가능한 발전을 추구해야 합니다. 기후 변화에 대한 적극적인 대응은 우리 세대뿐만 아니라 미래 세대를 위한 책임이기도 합니다.


In [11]:
# 피드백 작성 (여기서는 사람이 수동으로 평가하여 문자열 작성한다고 가정)
feedback = (
    "피드백:\n"
    " - 서론에서 주제 소개가 부족합니다.\n"
    " - 원인에 대한 설명이 모호하며 구체적 예시와 근거를 제시할 통계자료가 없습니다.\n"
    " - 해결 방안을 최소 10가지 이상 제시해주세요.\n"
    "위 사항을 반영하여 에세이를 수정해주세요."
)
# 개선 프롬프트 구성
retry_prompt = essay1 + "\n\n" + feedback
response_essay2 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": retry_prompt}
    ],
    model="gpt-4o-mini",
    temperature=0
)
essay2 = response_essay2.choices[0].message.content
print("2차 에세이:\n", essay2)

2차 에세이:
 기후 변화는 현대 사회가 직면한 가장 심각한 환경 문제 중 하나로, 이는 지구의 평균 기온 상승, 해수면 상승, 극단적인 기상 현상 등의 형태로 나타나고 있습니다. 이러한 변화는 인류의 생존과 지속 가능한 발전에 중대한 위협을 가하고 있으며, 이를 해결하기 위한 긴급한 노력이 필요합니다. 기후 변화의 주된 원인은 인간 활동에 의해 발생하는 온실가스의 증가로, 특히 화석 연료의 연소, 산업 활동, 농업 및 임업 등에서 발생하는 이산화탄소(CO2), 메탄(CH4), 아산화질소(N2O) 등의 온실가스가 대기 중에 축적되어 지구의 열을 가두는 효과를 만들어냅니다. 예를 들어, 2020년 기준으로 전 세계 이산화탄소 배출량은 약 33억 톤에 달하며, 이는 산업화 이후 급격히 증가한 수치입니다.

기후 변화의 해결 방안은 여러 가지가 있지만, 가장 중요한 것은 온실가스 배출을 줄이는 것입니다. 이를 위해 다음과 같은 10가지 이상의 구체적인 방안을 제시합니다.

1. **재생 가능 에너지 확대**: 태양광, 풍력, 수력 등 청정 에너지원으로의 전환을 통해 화석 연료 의존도를 줄이고, 온실가스 배출을 감소시켜야 합니다.

2. **에너지 효율성 향상**: 건물, 교통수단, 산업 공정에서 에너지를 절약하는 기술을 개발하고 보급하여 불필요한 에너지 소비를 줄여야 합니다.

3. **지속 가능한 농업 실천**: 유기농법, 회전농법 등을 통해 화학 비료와 농약 사용을 줄이고, 토양의 건강을 유지하여 탄소 흡수 능력을 높여야 합니다.

4. **임업 관리 개선**: 지속 가능한 산림 관리와 재조림을 통해 탄소를 흡수하고 생물 다양성을 보호해야 합니다.

5. **대중교통 이용 장려**: 대중교통 시스템을 개선하고 자전거 도로를 확충하여 개인 차량 사용을 줄이고, 교통에서 발생하는 온실가스를 감소시켜야 합니다.

6. **전기차 및 친환경 차량 보급**: 전기차와 수소차 등 친환경 차량의 보급을 확대하여 교통 부문에서의 온실가스 배출을 줄여야 합니다.

7. **폐기

In [12]:
# 피드백 작성 (여기서는 사람이 수동으로 평가하여 문자열 작성한다고 가정)
feedback_2 = (
    "피드백:\n"
    " - 관련 사실을 실증할만한 근거의 통계 자료가 없습니다.\n"
    "위 사항을 반영하여 에세이를 수정해주세요."
)
# 개선 프롬프트 구성
retry_prompt_2 = essay1 + "\n\n" + feedback_2
response_essay3 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": retry_prompt_2}
    ],
    model="gpt-4o-mini",
    temperature=0
)
essay3 = response_essay3.choices[0].message.content
print("3차 에세이:\n", essay3)

3차 에세이:
 기후 변화는 현대 사회가 직면한 가장 심각한 환경 문제 중 하나로, 지구의 평균 기온 상승, 해수면 상승, 극단적인 기상 현상 등의 형태로 나타나고 있습니다. 2021년 IPCC(기후변화에 관한 정부 간 패널) 보고서에 따르면, 지구의 평균 기온은 산업화 이전 수준보다 약 1.1도 상승했으며, 이 추세가 지속될 경우 2100년까지 1.5도 상승할 가능성이 높다고 경고하고 있습니다. 이러한 기후 변화의 주된 원인은 인간 활동에 의해 발생하는 온실가스의 증가입니다. 특히, 화석 연료의 연소, 산업 활동, 농업 및 임업 등에서 발생하는 이산화탄소(CO2), 메탄(CH4), 아산화질소(N2O) 등의 온실가스가 대기 중에 축적되어 지구의 열을 가두는 효과를 만들어냅니다. 2020년 기준으로, 전 세계 온실가스 배출량은 약 59.1억 톤 CO2에 달하며, 이 중 약 73%가 에너지 부문에서 발생하고 있습니다.

기후 변화의 해결 방안은 여러 가지가 있지만, 가장 중요한 것은 온실가스 배출을 줄이는 것입니다. 이를 위해 첫째, 재생 가능 에너지의 사용을 확대해야 합니다. 2020년 세계 에너지 기구(IEA)의 보고서에 따르면, 재생 가능 에너지원의 비율은 전 세계 전력 생산의 약 29%를 차지하고 있으며, 이는 2010년 대비 두 배 이상 증가한 수치입니다. 태양광, 풍력, 수력 등 청정 에너지원으로의 전환은 화석 연료 의존도를 줄이고, 온실가스 배출을 감소시키는 데 기여할 수 있습니다.

둘째, 에너지 효율성을 높이는 기술을 개발하고 보급해야 합니다. 미국 에너지부의 자료에 따르면, 에너지 효율성을 높이는 기술을 도입한 경우, 건물에서의 에너지 소비를 평균 30%까지 줄일 수 있습니다. 교통수단과 산업 공정에서도 에너지 절약 방법을 도입함으로써 불필요한 에너지 소비를 줄일 수 있습니다.

셋째, 지속 가능한 농업과 임업 practices를 통해 탄소 흡수 능력을 높이는 것도 중요합니다. 2019년 FAO(유엔 식량 농업 기구)의 보고서에 따르면, 지속 가